## Using Python to Perform Extract-Transform-Load (ETL Processing)
Modern Data Warehousing and Analytics solutions frequently use languages like Python or Scala to extract data from numerous sources, including relational database management systems, NoSQL database systems, real-time streaming endpoints and Data Lakes.  These languages can then be used to perform many types of transformation before then loading the data into a variety of destinations including file systems and data warehouses. This data can then be consumed by data scientists or business analysts.

In this lab you will recreate the **Northwind_DW** dimensional database from Lab 2; however, you'll take an entirely different approach. Instead of extracting, transforming and loading the date entirely on the database system entirely using SQL data definition language (DDL) and data manipulation language (DML) statements, here you will learn to interact with the RDBMS from a remote client running Python. You will learn to fetch data into Pandas DataFrames, perform all the necessary transformations in-memory on the client, and then push the newly transformed DataFrame back to the RDBMS using a Pandas function that will create the table and fill it with data with a single operation.

### Prerequisites:
#### Import the Necessary Libraries

In [1]:
import os
import numpy
import pandas as pd
import sqlalchemy as sqlal
from sqlalchemy import create_engine

#### Declare & Assign Connection Variables for the MySQL Server & Databases with which You'll be Working 

In [2]:
host_name = "localhost"
port = "3306"
user_id = "root"
pwd = "Passw0rd123"

src_dbname = "northwind"
dst_dbname = "northwind_dw_py"

#### Define Functions for Getting Data From and Setting Data Into Databases

In [3]:
def get_dataframe(user_id, pwd, host_name, db_name, sql_query):
    conn_str = f"mysql+pymysql://{user_id}:{pwd}@{host_name}/{db_name}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    connection = sqlEngine.connect()
    dframe = pd.read_sql(sql_query, connection)
    connection.close()
    
    return dframe


def set_dataframe(user_id, pwd, host_name, db_name, df, table_name, pk_column, db_operation):
    conn_str = f"mysql+pymysql://{user_id}:{pwd}@{host_name}/{db_name}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    connection = sqlEngine.connect()
    if db_operation == "insert":
        df.to_sql(table_name, con=connection, index=False, if_exists='replace')
        connection.execute(sqlal.text(f"ALTER TABLE {table_name} ADD PRIMARY KEY ({pk_column});"))
    elif db_operation == "update":
        df.to_sql(table_name, con=connection, index=False, if_exists='append')
    
    connection.close()

#### Create the New Data Warehouse database, and to Use it, Switch the Connection Context.
Clearly, you won't get very far without having a database to work with. Here we demonstrate how we can *drop* a database if it already exists, and then *create* the new **northwind_dw2** database and *use* it as the target of all subsequent operations.

In [4]:
conn_str = f"mysql+pymysql://{user_id}:{pwd}@{host_name}"
sqlEngine = create_engine(conn_str, pool_recycle=3600)
conn = sqlEngine.connect()
conn.execute(sqlal.text(f"DROP DATABASE IF EXISTS `{dst_dbname}`;"))
conn.execute(sqlal.text(f"CREATE DATABASE `{dst_dbname}`;"))
conn.execute(sqlal.text(f"USE {dst_dbname};"))
conn.close()

### 1.0. Create & Populate the Dimension Tables
At this point, we have to execute the script for **Lab 2c** which creates and populates a **Date Dimension** table.  Be certain to target this script to the new data warehouse database we just created in the preceding cell.  Later in this notebook we will integrate the **dim_date** table with the fact table by performing **lookup operations** to retreive the surrogate primary keys from the date dimension table that correspond with each **date** typed column in the fact table (e.g., order_date, paid_date, shipped_date).

#### 1.1. Extract Data from the Source Database Tables

In [5]:
sql_customers = sqlal.text("SELECT * FROM northwind.customers;")
df_customers = get_dataframe(user_id, pwd, host_name, src_dbname, sql_customers)
df_customers.head(2)

,id,company,last_name,first_name,email_address,job_title,business_phone,home_phone,mobile_phone,fax_number,address,city,state_province,zip_postal_code,country_region,web_page,notes,attachments
0,1,Company A,Bedecs,Anna,None,Owner,(123)555-0100,None,None,(123)555-0101,123 1st Street,Seattle,WA,99999,USA,None,None,b''
1,2,Company B,Gratacos Solsona,Antonio,None,Owner,(123)555-0100,None,None,(123)555-0101,123 2nd Street,Boston,MA,99999,USA,None,None,b''


In [6]:
sql_employees = sqlal.text("SELECT * FROM northwind.employees;")
df_employees = get_dataframe(user_id, pwd, host_name, src_dbname, sql_employees)
df_employees.head(2)

,id,company,last_name,first_name,email_address,job_title,business_phone,home_phone,mobile_phone,fax_number,address,city,state_province,zip_postal_code,country_region,web_page,notes,attachments
0,1,Northwind Traders,Freehafer,Nancy,nancy@northwindtraders.com,Sales Representative,(123)555-0100,(123)555-0102,None,(123)555-0103,123 1st Avenue,Seattle,WA,99999,USA,#http://northwindtraders.com#,None,b''
1,2,Northwind Traders,Cencini,Andrew,andrew@northwindtraders.com,"Vice President, Sales",(123)555-0100,(123)555-0102,None,(123)555-0103,123 2nd Avenue,Bellevue,WA,99999,USA,http://northwindtraders.com#http://northwindtr...,"Joined the company as a sales representative, ...",b''


In [7]:
sql_products = sqlal.text("SELECT * FROM northwind.products;")
df_products = get_dataframe(user_id, pwd, host_name, src_dbname, sql_products)
df_products.head(2)

,supplier_ids,id,product_code,product_name,description,standard_cost,list_price,reorder_level,target_level,quantity_per_unit,discontinued,minimum_reorder_quantity,category,attachments
0,4,1,NWTB-1,Northwind Traders Chai,None,13.5,18.0,10,40,10 boxes x 20 bags,0,10.0,Beverages,b''
1,10,3,NWTCO-3,Northwind Traders Syrup,None,7.5,10.0,25,100,12 - 550 ml bottles,0,25.0,Condiments,b''


In [8]:
sql_shippers = sqlal.text("SELECT * FROM northwind.shippers;")
df_shippers = get_dataframe(user_id, pwd, host_name, src_dbname, sql_shippers)
df_shippers.head(2)

,id,company,last_name,first_name,email_address,job_title,business_phone,home_phone,mobile_phone,fax_number,address,city,state_province,zip_postal_code,country_region,web_page,notes,attachments
0,1,Shipping Company A,None,None,None,None,None,None,None,None,123 Any Street,Memphis,TN,99999,USA,None,None,b''
1,2,Shipping Company B,None,None,None,None,None,None,None,None,123 Any Street,Memphis,TN,99999,USA,None,None,b''


#### 1.2. Perform Any Necessary Transformations
Pandas DataFrames enable extensive data modification capabilities. Here we will start by simply dropping features (columns) that we don't believe provide any real value to our analytics solution. Examples include columns having a high percentage of NULL values, columns having large amounts of free-text, and columns having binary large object (BLOB) data such as images or other documents. Then, we will rename the primary key column (id) to conform with data warehouse design standards.

In [9]:
drop_cols = ['email_address','home_phone','mobile_phone','web_page','notes','attachments']
df_customers.drop(drop_cols, axis=1, inplace=True)
df_customers.rename(columns={"id":"customer_key"}, inplace=True)
df_customers.head(2)

,customer_key,company,last_name,first_name,job_title,business_phone,fax_number,address,city,state_province,zip_postal_code,country_region
0,1,Company A,Bedecs,Anna,Owner,(123)555-0100,(123)555-0101,123 1st Street,Seattle,WA,99999,USA
1,2,Company B,Gratacos Solsona,Antonio,Owner,(123)555-0100,(123)555-0101,123 2nd Street,Boston,MA,99999,USA


In [10]:
drop_cols = ['mobile_phone','notes','attachments']
df_employees.drop(drop_cols, axis=1, inplace=True)
df_employees.rename(columns={"id":"employee_key"}, inplace=True)

df_employees.head(2)

,employee_key,company,last_name,first_name,email_address,job_title,business_phone,home_phone,fax_number,address,city,state_province,zip_postal_code,country_region,web_page
0,1,Northwind Traders,Freehafer,Nancy,nancy@northwindtraders.com,Sales Representative,(123)555-0100,(123)555-0102,(123)555-0103,123 1st Avenue,Seattle,WA,99999,USA,#http://northwindtraders.com#
1,2,Northwind Traders,Cencini,Andrew,andrew@northwindtraders.com,"Vice President, Sales",(123)555-0100,(123)555-0102,(123)555-0103,123 2nd Avenue,Bellevue,WA,99999,USA,http://northwindtraders.com#http://northwindtr...


In [11]:
drop_cols = ['supplier_ids','description','attachments']
df_products.drop(drop_cols, axis=1, inplace=True)
df_products.rename(columns={"id":"product_key"}, inplace=True)

df_products.head(2)

,product_key,product_code,product_name,standard_cost,list_price,reorder_level,target_level,quantity_per_unit,discontinued,minimum_reorder_quantity,category
0,1,NWTB-1,Northwind Traders Chai,13.5,18.0,10,40,10 boxes x 20 bags,0,10.0,Beverages
1,3,NWTCO-3,Northwind Traders Syrup,7.5,10.0,25,100,12 - 550 ml bottles,0,25.0,Condiments


In [12]:
drop_cols = ['last_name','first_name','email_address','job_title','business_phone',
             'home_phone','mobile_phone','fax_number','web_page','notes','attachments']
df_shippers.drop(drop_cols, axis=1, inplace=True)
df_shippers.rename(columns={"id":"shipper_key"}, inplace=True)
df_shippers.head(2)

,shipper_key,company,address,city,state_province,zip_postal_code,country_region
0,1,Shipping Company A,123 Any Street,Memphis,TN,99999,USA
1,2,Shipping Company B,123 Any Street,Memphis,TN,99999,USA


#### 1.4. Load the Transformed DataFrames into the New Data Warehouse by Creating New Tables
Here I demonstrate how we can create an iterable data structure containing the values needed to correctly create and populate the new dimension tables. If you inspect this code listing carefully, you'll notice that it's a **list** containing a **set** *(or vector)* for each dimension table. Each **set** then contains the *table_name* we need to assign to the table, the *pandas DataFrame* we crafted to define & populate the table, and the name we need to assign to the *primary_key* column.  With this *list of sets* defined, we can then call our **set_dataframe( )** function from within a **for *loop*** to create each *dimension* table.

In [13]:
db_operation = "insert"

tables = [('dim_customers', df_customers, 'customer_key'),
          ('dim_employees', df_employees, 'employee_key'),
          ('dim_products', df_products, 'product_key'),
          ('dim_shippers', df_shippers, 'shipper_key')]

In [14]:
for table_name, dataframe, primary_key in tables:
    set_dataframe(user_id, pwd, host_name, dst_dbname, dataframe, table_name, primary_key, db_operation)

### 2.0. Create & Populate the Fact Table
Here we will learn two approaches to creating the *fact_orders* fact table. The first approach demonstrates that a carefully crafted SQL SELECT statement can be used to perform this task... *but what fun would that be.* Seriously though, this approach is quick and effect if you already have the query, but what if you didn't have the opportunity to view and work with the data beforehand?  What's more, you may be required to combine data from multiple sources, some of which may not be relational database management systems. Then, a simple SQL query won't do!  You would need to load the data from the various sources (e.g., database tables, CSV or JSON files, NoSQL document collections, API stream data) and then combine them into a single dataframe that you could then use to create a new database table. For this reason we'll see how we can retrieve the data, but we won't bother to use it for creating a new table... we already know how to do that using the **set_dataframe( )** function anyway.

#### 2.1. First, you could simply use the SQL SELECT statement you authored in Lab 2 

In [15]:
fact_test = sqlal.text("""CREATE TABLE fact_orders (
  `order_key` int NOT NULL AUTO_INCREMENT,
  `order_id` int NOT NULL,
  `order_detail_id` int NOT NULL,
  `employee_id` int NOT NULL,
  `customer_id` int NOT NULL,
  `order_date` datetime,
  `paid_date_key` datetime,
  `shipped_date` datetime,
  `shipper_id` int,
  `shipping_fee` decimal(19,4),
  `quantity` decimal(18,4) NOT NULL DEFAULT '0.0000',
  `unit_price` decimal(19,4) DEFAULT '0.0000',
  `discount` double NOT NULL DEFAULT '0',
  `paid_date` datetime,
  `payment_type` varchar(50) DEFAULT NULL,
  `taxes` decimal(19,4),
  `tax_rate` double,
  `order_status` varchar(50),
  `order_details_status` varchar(50),
  PRIMARY KEY (`order_key`),
  KEY `order_id` (`order_id`),
  KEY `employee_id` (`employee_id`),
  KEY `customer_id` (`customer_id`),
  KEY `order_status` (`order_status`),
  KEY `shipper_id` (`shipper_id`)
) ENGINE=InnoDB AUTO_INCREMENT=82 DEFAULT CHARSET=utf8mb3;""")

#### 2.2. Instead, Implement the solution using Pandas DataFrames to Craft the table
This is where we get to the point of this lab.  We'll query the source **northwind** database to fill a *dataframe* for each of the source tables we need to create our *fact_orders* fact table; orders, orders_status, order_details and order_details_status. Then, we'll learn how to *join* those *dataframes* using the **merge( )** method of the Pandas DataFrame.  We'll make any additional changes that we expect to see reflected in the *fact* table in our new MySQL database, and then, we'll push the *dataframe* back to the MySQL server to create and populate the new *fact* table.

##### 2.2.1. Get all the data from each of the four tables involved

In [16]:
sql_orders = sqlal.text("SELECT * FROM northwind.orders;")
df_orders = get_dataframe(user_id,pwd,host_name,src_dbname,sql_orders)
df_orders.rename(columns={"id":"order_id"},inplace=True)
df_orders.head(2)

,order_id,employee_id,customer_id,order_date,shipped_date,shipper_id,ship_name,ship_address,ship_city,ship_state_province,ship_zip_postal_code,ship_country_region,shipping_fee,taxes,payment_type,paid_date,notes,tax_rate,tax_status_id,status_id
0,30,9,27,2006-01-15,2006-01-22,2.0,Karen Toh,789 27th Street,Las Vegas,NV,99999,USA,200.0,0.0,Check,2006-01-15,None,0.0,None,3
1,31,3,4,2006-01-20,2006-01-22,1.0,Christina Lee,123 4th Street,New York,NY,99999,USA,5.0,0.0,Credit Card,2006-01-20,None,0.0,None,3


In [17]:
sql_orders_status = sqlal.text("SELECT * FROM northwind.orders_status;")
df_orders_status = get_dataframe(user_id,pwd,host_name,src_dbname,sql_orders_status)
df_orders_status.rename(columns={"id":"status_id"},inplace=True)
df_orders_status.head(2)

,status_id,status_name
0,0,New
1,1,Invoiced


In [18]:
sql_order_details = sqlal.text("SELECT * FROM northwind.order_details;")
df_order_details = get_dataframe(user_id,pwd,host_name,src_dbname,sql_order_details)
df_order_details.rename(columns={"id":"order_details_id"},inplace=True)
df_order_details.head(2)

,order_details_id,order_id,product_id,quantity,unit_price,discount,status_id,date_allocated,purchase_order_id,inventory_id
0,27,30,34,100.0,14.0,0.0,2,None,96.0,83.0
1,28,30,80,30.0,3.5,0.0,2,None,NaN,63.0


In [19]:
sql_order_details_status = sqlal.text("SELECT * FROM northwind.order_details_status;")
df_order_details_status = get_dataframe(user_id,pwd,host_name,src_dbname,sql_order_details_status)
df_order_details_status.rename(columns={"id":"status_id"},inplace=True)
df_order_details_status.head(2)

,status_id,status_name
0,0,None
1,1,Allocated


##### 2.2.2. Get the order_status column.
Here we use the dataframe's **merge( )** method to **inner join** the *orders* and the *orders_status* dataframes **on** the *status_id* column. We then use the dataframe's **rename( )** method to rename the *status_name* column to *order_status*, and use the dataframe's **drop( )** method to remove the *status_id* column.

In [20]:
orders_merge = pd.merge(df_orders,df_orders_status,how='inner',on='status_id')
orders_merge.rename(columns={'status_name':'order_status'},inplace=True)
orders_merge = orders_merge.drop('status_id',axis=1)
orders_merge.head()

,order_id,employee_id,customer_id,order_date,shipped_date,shipper_id,ship_name,ship_address,ship_city,ship_state_province,ship_zip_postal_code,ship_country_region,shipping_fee,taxes,payment_type,paid_date,notes,tax_rate,tax_status_id,order_status
0,30,9,27,2006-01-15,2006-01-22,2.0,Karen Toh,789 27th Street,Las Vegas,NV,99999,USA,200.0,0.0,Check,2006-01-15,None,0.0,None,Closed
1,31,3,4,2006-01-20,2006-01-22,1.0,Christina Lee,123 4th Street,New York,NY,99999,USA,5.0,0.0,Credit Card,2006-01-20,None,0.0,None,Closed
2,32,4,12,2006-01-22,2006-01-22,2.0,John Edwards,123 12th Street,Las Vegas,NV,99999,USA,5.0,0.0,Credit Card,2006-01-22,None,0.0,None,Closed
3,33,6,8,2006-01-30,2006-01-31,3.0,Elizabeth Andersen,123 8th Street,Portland,OR,99999,USA,50.0,0.0,Credit Card,2006-01-30,None,0.0,None,Closed
4,34,9,4,2006-02-06,2006-02-07,3.0,Christina Lee,123 4th Street,New York,NY,99999,USA,4.0,0.0,Check,2006-02-06,None,0.0,None,Closed


##### 2.2.3. Get the order_details_status
Here we repeat the sequence of operations we used in the previous step to **inner join** the *order_details* and *order_details_status* dataframes for the sake of including the *order_details_status* column in place of the *status_id* column.

In [21]:
order_details_merge = pd.merge(df_order_details,df_order_details_status,how='inner',on='status_id')
order_details_merge.head(2)

,order_details_id,order_id,product_id,quantity,unit_price,discount,status_id,date_allocated,purchase_order_id,inventory_id,status_name
0,27,30,34,100.0,14.0,0.0,2,None,96.0,83.0,Invoiced
1,28,30,80,30.0,3.5,0.0,2,None,NaN,63.0,Invoiced


##### 2.2.4. Join the Orders and OrderDetails DataFrames
In this step we can now easily join the *orders* and *order_details* dataframes. Since each **order** (the *left* dataframe) can have many **order details** (the *right* dataframe), we'll need to implement a **right** *outer join* **on** the *order_id* column.

In [22]:
orders_orderdetails_merge = pd.merge(left=orders_merge,right=order_details_merge,how='right',on='order_id')
orders_orderdetails_merge.head()

,order_id,employee_id,customer_id,order_date,shipped_date,shipper_id,ship_name,ship_address,ship_city,ship_state_province,...,order_details_id,product_id,quantity,unit_price,discount,status_id,date_allocated,purchase_order_id,inventory_id,status_name
0,30,9,27,2006-01-15,2006-01-22,2.0,Karen Toh,789 27th Street,Las Vegas,NV,...,27,34,100.0,14.0,0.0,2,None,96.0,83.0,Invoiced
1,30,9,27,2006-01-15,2006-01-22,2.0,Karen Toh,789 27th Street,Las Vegas,NV,...,28,80,30.0,3.5,0.0,2,None,NaN,63.0,Invoiced
2,31,3,4,2006-01-20,2006-01-22,1.0,Christina Lee,123 4th Street,New York,NY,...,29,7,10.0,30.0,0.0,2,None,NaN,64.0,Invoiced
3,31,3,4,2006-01-20,2006-01-22,1.0,Christina Lee,123 4th Street,New York,NY,...,30,51,10.0,53.0,0.0,2,None,NaN,65.0,Invoiced
4,31,3,4,2006-01-20,2006-01-22,1.0,Christina Lee,123 4th Street,New York,NY,...,31,80,10.0,3.5,0.0,2,None,NaN,66.0,Invoiced


##### 2.2.5. Get the Data from the Date Dimension Table.
First, fetch the Surrogate Primary Key (date_key) and the Business Key (full_date) from the Date Dimension table using the **get_dataframe()** function. Also, be certain to cast the **full_date** column to the **datetime64** data type using the **.astype()** function that is native to Pandas DataFrame columns.

In [23]:
sql_date = sqlal.text("SELECT `date_key`,`full_date` FROM northwind_dw.dim_date;")
dw_dbname = "northwind_dw"
df_date = get_dataframe(user_id, pwd, host_name, dw_dbname, sql_date)
df_date.full_date = df_date.full_date.astype('datetime64')
df_date.head()

,date_key,full_date
0,20000101,2000-01-01
1,20000102,2000-01-02
2,20000103,2000-01-03
3,20000104,2000-01-04
4,20000105,2000-01-05


##### 2.2.6. Lookup the DateKeys from the Date Dimension Table.
Next, for each date typed column in the fact table, lookup the corresponding Surrogate Primary Key column.

In [24]:
# Lookup the Surrogate Primary Key (date_key) that Corresponds to the "order_date" Column.
#Create a dictionary for date_key and full_date pairs
date_key_mapping = df_date.set_index('full_date')['date_key'].to_dict()
#map dates to reference table
orders_orderdetails_merge['order_date'] = orders_orderdetails_merge['order_date'].map(date_key_mapping)
orders_orderdetails_merge.head()

,order_id,employee_id,customer_id,order_date,shipped_date,shipper_id,ship_name,ship_address,ship_city,ship_state_province,...,order_details_id,product_id,quantity,unit_price,discount,status_id,date_allocated,purchase_order_id,inventory_id,status_name
0,30,9,27,20060115.0,2006-01-22,2.0,Karen Toh,789 27th Street,Las Vegas,NV,...,27,34,100.0,14.0,0.0,2,None,96.0,83.0,Invoiced
1,30,9,27,20060115.0,2006-01-22,2.0,Karen Toh,789 27th Street,Las Vegas,NV,...,28,80,30.0,3.5,0.0,2,None,NaN,63.0,Invoiced
2,31,3,4,20060120.0,2006-01-22,1.0,Christina Lee,123 4th Street,New York,NY,...,29,7,10.0,30.0,0.0,2,None,NaN,64.0,Invoiced
3,31,3,4,20060120.0,2006-01-22,1.0,Christina Lee,123 4th Street,New York,NY,...,30,51,10.0,53.0,0.0,2,None,NaN,65.0,Invoiced
4,31,3,4,20060120.0,2006-01-22,1.0,Christina Lee,123 4th Street,New York,NY,...,31,80,10.0,3.5,0.0,2,None,NaN,66.0,Invoiced


In [25]:
# Lookup the Surrogate Primary Key (date_key) that Corresponds to the "paid_date" Column.
orders_orderdetails_merge.paid_date = orders_orderdetails_merge.paid_date.astype('datetime64[ns]')
#map dates to reference table
orders_orderdetails_merge['paid_date'] = orders_orderdetails_merge['paid_date'].map(date_key_mapping)
orders_orderdetails_merge.head()

,order_id,employee_id,customer_id,order_date,shipped_date,shipper_id,ship_name,ship_address,ship_city,ship_state_province,...,order_details_id,product_id,quantity,unit_price,discount,status_id,date_allocated,purchase_order_id,inventory_id,status_name
0,30,9,27,20060115.0,2006-01-22,2.0,Karen Toh,789 27th Street,Las Vegas,NV,...,27,34,100.0,14.0,0.0,2,None,96.0,83.0,Invoiced
1,30,9,27,20060115.0,2006-01-22,2.0,Karen Toh,789 27th Street,Las Vegas,NV,...,28,80,30.0,3.5,0.0,2,None,NaN,63.0,Invoiced
2,31,3,4,20060120.0,2006-01-22,1.0,Christina Lee,123 4th Street,New York,NY,...,29,7,10.0,30.0,0.0,2,None,NaN,64.0,Invoiced
3,31,3,4,20060120.0,2006-01-22,1.0,Christina Lee,123 4th Street,New York,NY,...,30,51,10.0,53.0,0.0,2,None,NaN,65.0,Invoiced
4,31,3,4,20060120.0,2006-01-22,1.0,Christina Lee,123 4th Street,New York,NY,...,31,80,10.0,3.5,0.0,2,None,NaN,66.0,Invoiced


In [26]:
# Lookup the Surrogate Primary Key (date_key) that Corresponds to the "shipped_date" Column.
#map dates to reference table
orders_orderdetails_merge['shipped_date'] = orders_orderdetails_merge['shipped_date'].map(date_key_mapping)
orders_orderdetails_merge.head()

,order_id,employee_id,customer_id,order_date,shipped_date,shipper_id,ship_name,ship_address,ship_city,ship_state_province,...,order_details_id,product_id,quantity,unit_price,discount,status_id,date_allocated,purchase_order_id,inventory_id,status_name
0,30,9,27,20060115.0,20060122.0,2.0,Karen Toh,789 27th Street,Las Vegas,NV,...,27,34,100.0,14.0,0.0,2,None,96.0,83.0,Invoiced
1,30,9,27,20060115.0,20060122.0,2.0,Karen Toh,789 27th Street,Las Vegas,NV,...,28,80,30.0,3.5,0.0,2,None,NaN,63.0,Invoiced
2,31,3,4,20060120.0,20060122.0,1.0,Christina Lee,123 4th Street,New York,NY,...,29,7,10.0,30.0,0.0,2,None,NaN,64.0,Invoiced
3,31,3,4,20060120.0,20060122.0,1.0,Christina Lee,123 4th Street,New York,NY,...,30,51,10.0,53.0,0.0,2,None,NaN,65.0,Invoiced
4,31,3,4,20060120.0,20060122.0,1.0,Christina Lee,123 4th Street,New York,NY,...,31,80,10.0,3.5,0.0,2,None,NaN,66.0,Invoiced


##### 2.2.5. Perform any Additional Transformations
In this step we can prepare the DataFrame so that it defines exactly what we want to see created in the database.  Issues may include dropping unwanted columns, reordering the columns, and in our case, creating a new column to serve as the primary key.

In [27]:
#Lets look at columns
orders_orderdetails_merge.columns

Index(['order_id', 'employee_id', 'customer_id', 'order_date', 'shipped_date',
       'shipper_id', 'ship_name', 'ship_address', 'ship_city',
       'ship_state_province', 'ship_zip_postal_code', 'ship_country_region',
       'shipping_fee', 'taxes', 'payment_type', 'paid_date', 'notes',
       'tax_rate', 'tax_status_id', 'order_status', 'order_details_id',
       'product_id', 'quantity', 'unit_price', 'discount', 'status_id',
       'date_allocated', 'purchase_order_id', 'inventory_id', 'status_name'],
      dtype='object')

In [28]:
columns_to_keep = ['order_id','order_details_id','employee_id','customer_id','order_date','paid_date','shipped_date','shipper_id','shipping_fee','quantity','unit_price','discount','payment_type','taxes','tax_rate','status_name']
#Create final df. By making a new, separate df, we avoid losing forever columns that we may want to come back to in the future.
fact_orders = orders_orderdetails_merge[columns_to_keep]
#Create primary key called id
fact_orders['id'] = fact_orders.index
id = fact_orders.pop('id')
fact_orders.insert(0,'id',id)
fact_orders.reset_index(drop=True, inplace=True)
fact_orders.head()

/var/folders/36/j7ky89tj7hlb6vw40hsl12xh0000gn/T/ipykernel_11931/327566297.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fact_orders['id'] = fact_orders.index


,id,order_id,order_details_id,employee_id,customer_id,order_date,paid_date,shipped_date,shipper_id,shipping_fee,quantity,unit_price,discount,payment_type,taxes,tax_rate,status_name
0,0,30,27,9,27,20060115.0,20060115.0,20060122.0,2.0,200.0,100.0,14.0,0.0,Check,0.0,0.0,Invoiced
1,1,30,28,9,27,20060115.0,20060115.0,20060122.0,2.0,200.0,30.0,3.5,0.0,Check,0.0,0.0,Invoiced
2,2,31,29,3,4,20060120.0,20060120.0,20060122.0,1.0,5.0,10.0,30.0,0.0,Credit Card,0.0,0.0,Invoiced
3,3,31,30,3,4,20060120.0,20060120.0,20060122.0,1.0,5.0,10.0,53.0,0.0,Credit Card,0.0,0.0,Invoiced
4,4,31,31,3,4,20060120.0,20060120.0,20060122.0,1.0,5.0,10.0,3.5,0.0,Credit Card,0.0,0.0,Invoiced


##### 2.2.6. Write the DataFrame Back to the Database

In [29]:
db_operation = "insert"
tables = [('fact_orders', fact_orders, 'id')]
for table_name, dataframe, primary_key in tables:
    set_dataframe(user_id, pwd, host_name, dst_dbname, dataframe, table_name, primary_key, db_operation)

### 3.0. Demonstrate that the New Data Warehouse Exists and Contains the Correct Data
To demonstrate the viability of your solution, author a SQL SELECT statement that returns:
-	Each Customer’s Last Name
-	The total amount of the order quantity associated with each customer
-	The total amount of the order unit price associated with each customer

In [30]:
#Each customer's last name
sql_customername = sqlal.text("SELECT `last_name` FROM northwind_dw_py.dim_customers;")
dw_dbname = 'northwind_dw_py'
df_customername = get_dataframe(user_id, pwd, host_name, dw_dbname, sql_customername)
df_customername


,last_name
0,Bedecs
1,Gratacos Solsona
2,Axen
3,Lee
4,O’Donnell
5,Pérez-Olaeta
6,Xie
7,Andersen
8,Mortensen
9,Wacker


In [31]:

#Total amount of the order quantity associated with each customer
sql_customerquantity = sqlal.text(
    "SELECT customer_id, SUM(quantity) AS quantity_sum "
    "FROM northwind_dw_py.fact_orders "
    "GROUP BY customer_id"
)
df_customerquantity = get_dataframe(user_id, pwd, host_name, dw_dbname, sql_customerquantity)
df_customerquantity


,customer_id,quantity_sum
0,27,130.0
1,4,140.0
2,12,35.0
3,8,260.0
4,29,137.0
5,3,253.0
6,6,427.0
7,28,405.0
8,10,265.0
9,9,160.0


In [32]:

#Total amount of the order unit price associated with each customer
sql_unitprice = sqlal.text(
    "SELECT customer_id, SUM(unit_price) AS unit_price_sum "
    "FROM northwind_dw_py.fact_orders "
    "GROUP BY customer_id"
)
df_unitprice = get_dataframe(user_id, pwd, host_name, dw_dbname, sql_unitprice)
df_unitprice

,customer_id,unit_price_sum
0,27,17.50
1,4,221.70
2,12,64.00
3,8,118.70
4,29,65.75
5,3,100.64
6,6,162.50
7,28,120.05
8,10,72.69
9,9,63.95


### 3.1 Extra Credit: Author a Query that Returns the Total Shipping Fee per Shipper

In [34]:
shipping_fee_query = sqlal.text(
"""
SELECT d.company as company_name, SUM(o.shipping_fee) AS shipping_fee_sum
FROM northwind_dw_py.dim_shippers d
JOIN northwind_dw_py.fact_orders o ON d.shipper_key = o.shipper_id
GROUP BY company_name
"""
)
df_shippingfee = get_dataframe(user_id, pwd, host_name, dw_dbname, shipping_fee_query)
df_shippingfee

,company_name,shipping_fee_sum
0,Shipping Company B,1618.0
1,Shipping Company A,335.0
2,Shipping Company C,479.0
